Import Necessary Libraries

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

import warnings
warnings.filterwarnings('ignore')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Load data

In [33]:
df = pd.read_csv('https://raw.githubusercontent.com/vikashishere/YT-MLOPS-Complete-ML-Pipeline/refs/heads/main/experiments/spam.csv', usecols=['v1', 'v2'])
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [34]:
df.columns = ['target', 'text']
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [35]:
df.shape

(5572, 2)

In [17]:
df.isnull().sum().sum()/df.size * 100

np.float64(0.0)

In [20]:
df.duplicated().sum()

np.int64(403)

In [36]:
df[df.duplicated()]

,target,text
102,ham,As per your request 'Melle Melle (Oru Minnamin...
153,ham,As per your request 'Melle Melle (Oru Minnamin...
206,ham,"As I entered my cabin my PA said, '' Happy B'd..."
222,ham,"Sorry, I'll call later"
325,ham,No calls..messages..missed calls
...,...,...
5524,spam,You are awarded a SiPix Digital Camera! call 0...
5535,ham,"I know you are thinkin malaria. But relax, chi..."
5539,ham,Just sleeping..and surfing
5553,ham,Hahaha..use your brain dear


In [37]:
clean_df = df.drop_duplicates()
clean_df.shape

(5169, 2)

In [22]:
clean_df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [38]:
clean_df.duplicated().sum()

np.int64(0)

Data Preprocessing

In [39]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
clean_df['target'] = le.fit_transform(clean_df['target'])
clean_df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


Feature Engineering

In [40]:
from nltk.stem.porter import PorterStemmer
import string

ps = PorterStemmer()

In [41]:
from numpy._core.defchararray import isalnum
def transform_text(text):
  text = text.lower()
  text = nltk.word_tokenize(text)

  """
      Removing special characters and numbers
  """
  y = list()
  for i in text:
    if i.isalnum():
      y.append(i)

  # Removing stop words and punctuation
  text = y[:]
  y.clear()



  # Loops through the tokens and remove stopwords and punctuation
  for i in text:
    if i not in stopwords.words('english') and i not in string.punctuation:
      y.append(i)

  # Stemming using Porter Stemmer
  for i in text:
    y.append(ps.stem(i))

  # Join the processed tokens back into a single string
  return " ".join(y)

In [44]:
transform_text("Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...")

'go jurong point crazy available bugis n great world la e buffet cine got amore wat go until jurong point crazi avail onli in bugi n great world la e buffet cine there got amor wat'

In [45]:
clean_df['transformed_text'] = clean_df.apply(lambda x: transform_text(x['text']), axis=1)
clean_df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,0,U dun say so early hor... U c already then say...,u dun say early hor u c already say u dun say ...
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goes usf lives around though nah i d...


In [46]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

tfid = TfidfVectorizer(max_features=3000)

In [49]:
# Seperating X and y
X = tfid.fit_transform(clean_df['transformed_text']).toarray()
y = clean_df.target.values

Train-Test Split

In [50]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

Model Training

In [51]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [53]:
models = {
    'Logistic Regression': LogisticRegression(solver = 'liblinear', penalty = 'l1'),
    'Support Vector Classifier': SVC(kernel = 'sigmoid', gamma = 1.0),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Decision Tree Classifier': DecisionTreeClassifier(max_depth = 5),
    'K Neighbors Classifier': KNeighborsClassifier(n_neighbors = 3),
    'Random Forest Classifier': RandomForestClassifier(n_estimators = 50, random_state = 42),
    'Ada Boost Classifier': AdaBoostClassifier(n_estimators = 50, random_state = 42),
    'Bagging Classifier': BaggingClassifier(n_estimators = 50, random_state = 42),
    'Extra Trees Classifier': ExtraTreesClassifier(n_estimators = 50, random_state = 42),
    'Gradient Boosting Classifier': GradientBoostingClassifier(n_estimators = 50, random_state = 42),
    'XGB Classifier': XGBClassifier(n_estimators = 50, random_state = 42)

}

Model Evaluation

In [59]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

def evaluate_model(X_train, X_test, y_train, y_test, models):
  report = dict()

  for model_name, model in models.items():
    model.fit(X_train, y_train)
    yhat = model.predict(X_test)

    accuracy = accuracy_score(y_test, yhat)
    precision = precision_score(y_test, yhat)
    recall = recall_score(y_test, yhat)
    f1 = f1_score(y_test, yhat)

    report[model_name] = {
        'accuracy': np.round((accuracy),3),
        'precision': np.round((precision),3),
        'recall': np.round((recall),3),
        'f1': np.round((f1),3)
    }

  return pd.DataFrame(report).T.sort_values(by = ['accuracy', 'precision', 'recall', 'f1'], ascending = False)


In [60]:
df = evaluate_model(X_train, X_test, y_train, y_test, models)
df

,accuracy,precision,recall,f1
Support Vector Classifier,0.982,0.970,0.897,0.932
Extra Trees Classifier,0.981,0.984,0.876,0.927
XGB Classifier,0.977,0.984,0.848,0.911
Multinomial Naive Bayes,0.976,0.992,0.834,0.906
Random Forest Classifier,0.974,0.984,0.828,0.899
Bagging Classifier,0.963,0.885,0.848,0.866
Logistic Regression,0.959,0.926,0.772,0.842
Gradient Boosting Classifier,0.955,0.962,0.703,0.813
Ada Boost Classifier,0.938,0.901,0.628,0.740
Decision Tree Classifier,0.938,0.846,0.683,0.756
